In [2]:
!pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 29.8 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- --

In [11]:
!pip install xgboost

   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   - -------------------------------------- 4.2/124.9 MB 41.8 MB/s eta 0:00:03
   ---- ----------------------------------- 13.1/124.9 MB 43.2 MB/s eta 0:00:03
   ------- -------------------------------- 24.6/124.9 MB 47.2 MB/s eta 0:00:03
   -------- ------------------------------- 27.8/124.9 MB 38.3 MB/s eta 0:00:03
   ---------- ----------------------------- 32.0/124.9 MB 33.3 MB/s eta 0:00:03
   ------------- -------------------------- 42.5/124.9 MB 36.0 MB/s eta 0:00:03
   --------------- ------------------------ 49.5/124.9 MB 36.7 MB/s eta 0:00:03
   ----------------- ---------------------- 53.2/124.9 MB 32.9 MB/s eta 0:00:03
   ------------------- -------------------- 61.1/124.9 MB 34.4 MB/s eta 0:00:02
   --------------------- ------------------ 66.6/124.9 MB 32.7 MB/s eta 0:00:02
   ------------------------ --------------- 77.6/124.9 MB 3

In [2]:
# =============================================================================
# 15분(15T) + Full 피처만 사용 (Optuna 없음)
# 목표: "공장 전체(합산) 기준"으로 datetime당 1행만 예측 파일 생성
#
# - 입력 학습 데이터(TRAIN_15MIN_PATH): 모듈별 15분 집계 파일이라도 OK
#   -> datetime 기준으로 공장 전체로 집계해서 학습
# - 템플릿(TEMPLATE_PATH): 모듈별 행이 있어도 OK
#   -> datetime unique만 뽑아서 datetime당 1행 템플릿을 새로 만든 뒤 채움
#
# 사용자 확정:
# 1) operation 예측 안 함 -> 1로 고정
# 2) 모든 변수 동일 모델 구조로 예측
# 3) accumActiveEnergy 0부터 시작
# 4) hourly_energy_used_kWh 컬럼명 유지, 값은 15분 kWh로 채움
# 5) 전력가중평균은 별도 컬럼(_wavg)로 추가
# =============================================================================

import os
import gc
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

# =========================
# 경로/상수 설정
# =========================
TRAIN_15MIN_PATH = r"D:/2025-2/BDA/공모전/sample data/rtu_data_15min_v2.csv"   # 모듈별 15분 집계(학습용)
TEMPLATE_PATH    = r"D:/2025-2/BDA/공모전/sample data/rtu_data_15min_v2.csv"   # 형식 기준(모듈별이어도 됨)
OUTPUT_PATH      = r"D:/2025-2/BDA/공모전/sample data/output/pred_total_15min_1row_per_datetime.csv"

PRICE_WON_PER_KWH = 180
CARBON_KGCO2_PER_KWH = 0.424
FREQ = "15T"
STEP_HOURS = 0.25  # 15분 = 0.25h

MODULE_COL = "module(equipment)"
DT_COL = "datetime"

# =========================
# 유틸
# =========================
def safe_mkdir(path: str):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)

def robust_parse_datetime(s: pd.Series) -> pd.DatetimeIndex:
    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        bad = int(dt.isna().sum())
        examples = s[dt.isna()].head(3).tolist()
        raise ValueError(f"[datetime parse] 실패 {bad}개. 예: {examples}")
    return pd.DatetimeIndex(dt)

def create_features_15m_full(index_dt: pd.DatetimeIndex) -> pd.DataFrame:
    df = pd.DataFrame(index=index_dt).copy()
    df["hour"] = df.index.hour
    df["dayofweek"] = df.index.dayofweek
    df["day"] = df.index.day
    df["month"] = df.index.month
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # 기존 로직 유지(일자 1~31)
    df["day_sin"] = np.sin(2 * np.pi * df["day"] / 31)
    df["day_cos"] = np.cos(2 * np.pi * df["day"] / 31)
    return df

def wavg(series, weights):
    mask = series.notna() & weights.notna()
    s = series[mask]
    w = weights[mask]
    w_sum = w.sum()
    if w_sum == 0 or len(s) == 0:
        return np.nan
    return float((s * w).sum() / w_sum)

# =========================
# 1) 학습 데이터 로드 -> 공장 전체 집계(df_total) 생성
# =========================
df_train = pd.read_csv(TRAIN_15MIN_PATH)

if DT_COL not in df_train.columns:
    raise KeyError(f"학습 파일에 {DT_COL} 컬럼이 없습니다.")
df_train[DT_COL] = robust_parse_datetime(df_train[DT_COL])

if "activePow" not in df_train.columns:
    raise KeyError("학습 파일에 activePow 컬럼이 없습니다.")

# 수치형 변환(집계 안정화)
for c in df_train.columns:
    if c not in [DT_COL, MODULE_COL]:
        df_train[c] = pd.to_numeric(df_train[c], errors="coerce")

SUM_COLS = [c for c in ["activePow", "currentR", "currentS", "currentT", "reactiveP"] if c in df_train.columns]
MEAN_COLS = [c for c in [
    "voltageR","voltageS","voltageT",
    "voltageRS","voltageST","voltageTR",
    "powerFactR","powerFactS","powerFactT"
] if c in df_train.columns]

WAVG_BASE_COLS = MEAN_COLS[:]  # 전력가중평균은 전압/역률에 대해 별도 컬럼 생성

rows = []
for dt, g in df_train.groupby(DT_COL):
    row = {DT_COL: dt}

    # 합: 전력/전류/무효전력
    for c in SUM_COLS:
        row[c] = float(g[c].sum(skipna=True))

    # 단순 평균: 전압/역률
    for c in MEAN_COLS:
        row[c] = float(g[c].mean(skipna=True))

    # 전력 가중 평균(가중치=모듈 activePow): 별도 컬럼
    for c in WAVG_BASE_COLS:
        row[f"{c}_wavg"] = wavg(g[c], g["activePow"])

    rows.append(row)

df_total = (pd.DataFrame(rows)
            .sort_values(DT_COL)
            .set_index(DT_COL)
            .sort_index()
            .resample(FREQ).mean())  # 15분 그리드 보정

# =========================
# 2) 학습 피처/타깃 정의 (15분 + Full 고정)
# =========================
FEATURE_COLS = ["hour","dayofweek","day","month","is_weekend","hour_sin","hour_cos","day_sin","day_cos"]
X_all = create_features_15m_full(df_total.index)[FEATURE_COLS]

BASE_TARGETS = [
    "activePow",
    "voltageR","voltageS","voltageT",
    "voltageRS","voltageST","voltageTR",
    "currentR","currentS","currentT",
    "powerFactR","powerFactS","powerFactT",
    "reactiveP"
]
TARGET_COLS = [c for c in BASE_TARGETS if c in df_total.columns]
TARGET_COLS += [f"{c}_wavg" for c in WAVG_BASE_COLS if f"{c}_wavg" in df_total.columns]

if "activePow" not in TARGET_COLS:
    raise KeyError("df_total에 activePow가 없습니다. (예측 필수 타깃)")

# =========================
# 3) 타깃별 모델 학습 (Optuna 없이 고정 파라미터)
# =========================
BASE_PARAMS = dict(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=64,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

models = {}
for t in TARGET_COLS:
    data = pd.concat([X_all, df_total[t].rename("y")], axis=1).dropna()
    if len(data) < 200:
        raise ValueError(f"[{t}] 학습 데이터가 너무 적습니다: {len(data)}행")

    X = data[FEATURE_COLS]
    y = data["y"]

    m = lgb.LGBMRegressor(**BASE_PARAMS)
    m.fit(X, y)
    models[t] = m

    gc.collect()

# =========================
# 4) 템플릿 -> datetime unique만 사용해 "datetime당 1행" 템플릿 생성
# =========================
df_template = pd.read_csv(TEMPLATE_PATH)
if DT_COL not in df_template.columns:
    raise KeyError(f"템플릿에 {DT_COL} 컬럼이 없습니다.")
dt_future = robust_parse_datetime(df_template[DT_COL])

# datetime unique (정렬)
dt_unique = pd.DatetimeIndex(pd.Series(dt_future).drop_duplicates().sort_values().to_list())

# 출력 템플릿(1행/시간)
# - 컬럼은 '템플릿과 동일'을 최대한 유지하되, 모듈 컬럼이 있으면 TOTAL로 고정
df_out = pd.DataFrame({DT_COL: dt_unique})
if MODULE_COL in df_template.columns:
    df_out[MODULE_COL] = "TOTAL"

# 템플릿의 나머지 컬럼을 존재시키기 위해, 일단 NaN으로 만들어둠(나중에 예측/파생으로 채움)
template_cols = df_template.columns.tolist()
for c in template_cols:
    if c not in df_out.columns:
        df_out[c] = np.nan

# =========================
# 5) 예측 수행(모든 타깃) + 파생/누적 컬럼 생성
# =========================
X_future = create_features_15m_full(dt_unique)[FEATURE_COLS]

# 타깃 예측값 채우기
for t, m in models.items():
    yhat = m.predict(X_future)
    if t == "activePow":
        yhat = np.maximum(yhat, 0)
    df_out[t] = yhat

# operation: 예측 안 함 -> 1 고정(템플릿에 있으면)
if "operation" in df_out.columns:
    df_out["operation"] = 1

# 15분 kWh (컬럼명 유지)
df_out["kWh_from_activePow"] = df_out["activePow"] * STEP_HOURS

if "hourly_energy_used_kWh" in df_out.columns:
    # 컬럼명은 그대로 두되 값은 15분 kWh
    df_out["hourly_energy_used_kWh"] = df_out["kWh_from_activePow"]

# accumActiveEnergy: 0부터 누적
if "accumActiveEnergy" in df_out.columns:
    df_out["accumActiveEnergy"] = df_out["kWh_from_activePow"].cumsum()

# SumCurrent / MeanPowerFactor
if all(c in df_out.columns for c in ["currentR","currentS","currentT"]):
    df_out["SumCurrent"] = df_out["currentR"] + df_out["currentS"] + df_out["currentT"]

if all(c in df_out.columns for c in ["powerFactR","powerFactS","powerFactT"]):
    df_out["MeanPowerFactor"] = (df_out["powerFactR"] + df_out["powerFactS"] + df_out["powerFactT"]) / 3

# 비용/탄소 (예측 기반)
if "electricityCost_won" in df_out.columns:
    df_out["electricityCost_won"] = df_out["kWh_from_activePow"] * PRICE_WON_PER_KWH
if "carbon_kgCO2" in df_out.columns:
    df_out["carbon_kgCO2"] = df_out["kWh_from_activePow"] * CARBON_KGCO2_PER_KWH

# =========================
# 6) 컬럼 순서: 템플릿 컬럼 우선 유지 + 추가된 _wavg 컬럼은 뒤에 붙임
# =========================
extra_cols = [c for c in df_out.columns if c not in template_cols]
df_out = df_out[template_cols + extra_cols]

safe_mkdir(OUTPUT_PATH)
df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("saved:", OUTPUT_PATH)
print("rows (unique datetimes):", len(df_out))
print("added extra cols (wavg etc.):", extra_cols)


saved: D:/2025-2/BDA/공모전/sample data/output/pred_total_15min_1row_per_datetime.csv
rows (unique datetimes): 14401
added extra cols (wavg etc.): ['voltageR_wavg', 'voltageS_wavg', 'voltageT_wavg', 'voltageRS_wavg', 'voltageST_wavg', 'voltageTR_wavg', 'powerFactR_wavg', 'powerFactS_wavg', 'powerFactT_wavg']


In [5]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

# =============================================================================
# 경로 설정
# =============================================================================
RTU_15MIN_PATH = r"D:/2025-2/BDA/공모전/sample data/rtu_data_15min_v2.csv"          # 모듈별 15분 데이터(학습용)
ROW_5MONTH_PATH = r"D:/2025-2/BDA/공모전/sample data/rtu_data_5month_dynamic_fixed.csv"  # 행(시간)만 제공
OUTPUT_PATH = r"D:/2025-2/BDA/공모전/sample data/output/pred_total_rows_5month_cols_rtu15min.csv"

PRICE_WON_PER_KWH = 180
CARBON_KGCO2_PER_KWH = 0.424
STEP_HOURS = 0.25  # 15분

DT_CANDIDATES = ("datetime", "id", "timestamp", "localtime")
MODULE_COL = "module(equipment)"

# =============================================================================
# 유틸
# =============================================================================
def safe_mkdir(path: str):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)

def find_dt_col(df: pd.DataFrame) -> str:
    for c in DT_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError(f"시간 컬럼을 찾지 못했습니다. 후보={DT_CANDIDATES}, 실제={df.columns.tolist()}")

def to_dt_index(s: pd.Series) -> pd.DatetimeIndex:
    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        bad = int(dt.isna().sum())
        ex = s[dt.isna()].head(3).tolist()
        raise ValueError(f"datetime 파싱 실패 {bad}개. 예: {ex}")
    return pd.DatetimeIndex(dt)

def time_features(index_dt: pd.DatetimeIndex) -> pd.DataFrame:
    """15분 1행 예측을 위해 minute, 요일 주기(dow) 포함"""
    df = pd.DataFrame(index=index_dt).copy()

    df["hour"] = df.index.hour
    df["minute"] = df.index.minute
    df["dayofweek"] = df.index.dayofweek
    df["day"] = df.index.day
    df["month"] = df.index.month
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

    df["minute_sin"] = np.sin(2*np.pi*df["minute"]/60)
    df["minute_cos"] = np.cos(2*np.pi*df["minute"]/60)

    # day-of-month(기존 흐름 유지)
    df["day_sin"] = np.sin(2*np.pi*df["day"]/31)
    df["day_cos"] = np.cos(2*np.pi*df["day"]/31)

    # day-of-week(요청 3)
    df["dow_sin"] = np.sin(2*np.pi*df["dayofweek"]/7)
    df["dow_cos"] = np.cos(2*np.pi*df["dayofweek"]/7)

    return df

def add_lag_rolling(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """요청 1: lag/rolling 추가 (공장 전체 시계열 1개 기준)"""
    out = pd.DataFrame(index=df.index)
    s = df[col]

    out[f"{col}_lag1"] = s.shift(1)       # 15분 전
    out[f"{col}_lag4"] = s.shift(4)       # 1시간 전
    out[f"{col}_lag96"] = s.shift(96)     # 1일 전 (15분*96)
    out[f"{col}_rmean4"] = s.rolling(4, min_periods=1).mean()
    out[f"{col}_rstd4"]  = s.rolling(4, min_periods=2).std()
    out[f"{col}_rmean96"] = s.rolling(96, min_periods=1).mean()

    return out

# =============================================================================
# 1) rtu_15min(모듈별) -> 공장 전체(합산/평균 규칙) 시계열 생성
# =============================================================================
df_rtu = pd.read_csv(RTU_15MIN_PATH)
rtu_cols = df_rtu.columns.tolist()

dt_col_rtu = find_dt_col(df_rtu)
df_rtu["__dt__"] = to_dt_index(df_rtu[dt_col_rtu])

# 수치형 캐스팅
for c in df_rtu.columns:
    if c not in [dt_col_rtu, "__dt__", MODULE_COL]:
        df_rtu[c] = pd.to_numeric(df_rtu[c], errors="coerce")

# 집계 규칙(공장 전체)
SUM_COLS = [c for c in ["activePow", "currentR", "currentS", "currentT", "reactiveP"] if c in df_rtu.columns]
MEAN_COLS = [c for c in [
    "voltageR","voltageS","voltageT",
    "voltageRS","voltageST","voltageTR",
    "powerFactR","powerFactS","powerFactT",
] if c in df_rtu.columns]

agg = {}
for c in SUM_COLS:
    agg[c] = "sum"
for c in MEAN_COLS:
    agg[c] = "mean"

df_total = (
    df_rtu.groupby("__dt__", as_index=False)
          .agg(agg)
          .sort_values("__dt__")
          .set_index("__dt__")
          .sort_index()
          .resample("15T").mean()
)

if "activePow" not in df_total.columns:
    raise KeyError("공장 전체 시계열에 activePow가 없습니다. 원본 컬럼명을 확인하세요.")

# =============================================================================
# 2) 학습 피처 구성(시간피처 + lag/rolling)
# =============================================================================
X_time = time_features(df_total.index)
X_lag  = add_lag_rolling(df_total, "activePow")
X_all  = pd.concat([X_time, X_lag], axis=1)

FEATURE_COLS = X_all.columns.tolist()

# 예측 대상(공장 전체 기준)
BASE_TARGETS = [
    "activePow",
    "voltageR","voltageS","voltageT",
    "voltageRS","voltageST","voltageTR",
    "currentR","currentS","currentT",
    "powerFactR","powerFactS","powerFactT",
    "reactiveP",
]
TARGET_COLS = [c for c in BASE_TARGETS if c in df_total.columns]

# =============================================================================
# 3) 타깃별 모델 학습(Optuna 없음)
# =============================================================================
LGB_PARAMS = dict(
    n_estimators=1600,
    learning_rate=0.05,
    max_depth=12,
    num_leaves=96,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

models = {}
for t in TARGET_COLS:
    y = df_total[t]
    data = pd.concat([X_all, y.rename("y")], axis=1).dropna()

    if len(data) < 500:
        raise ValueError(f"[{t}] 학습 데이터가 너무 적습니다: {len(data)}행 (lag로 인해 앞부분 drop 가능)")

    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(data[FEATURE_COLS], data["y"])
    models[t] = m
    gc.collect()

# =============================================================================
# 4) 5month_dynamic에서 "행(시간)" 가져오기
# =============================================================================
df_rows = pd.read_csv(ROW_5MONTH_PATH)
dt_col_rows = find_dt_col(df_rows)
dt_future = to_dt_index(df_rows[dt_col_rows])

# 행은 5month의 "원래 순서"를 유지하되,
# lag/rolling 계산을 위해 내부적으로는 시간 정렬 버전도 만든 뒤 다시 원순서로 되돌립니다.
df_future = pd.DataFrame({"__dt__": dt_future})
df_future["__ord__"] = np.arange(len(df_future))

df_future_sorted = df_future.sort_values("__dt__").reset_index(drop=True)

# =============================================================================
# 5) 예측: 공장 전체 15분 1행씩 예측
#    - lag/rolling은 "activePow"가 필요하므로,
#      activePow는 재귀(recursive)로 먼저 예측해서 lag를 업데이트
# =============================================================================
# 준비: 과거 activePow 시계열을 이어붙여 초기 lag를 만들 기반 확보
hist_active = df_total["activePow"].copy()

# future 예측용 컨테이너(정렬된 시간 기준)
pred_sorted = pd.DataFrame(index=df_future_sorted["__dt__"])
pred_sorted.index.name = "__dt__"

# 5-1) activePow 재귀 예측
#     각 타임스텝에서 (time features + lag features computed from "hist+pred")로 1스텝 예측
pred_active = []
for ts in pred_sorted.index:
    # 현재까지의 시계열(과거 + 지금까지 예측)
    series_now = pd.concat([hist_active, pd.Series(pred_active, index=pred_sorted.index[:len(pred_active)])])

    # 필요한 lag/rolling을 얻기 위해 ts 직전까지 포함하는 series로 계산
    tmp_df = pd.DataFrame({"activePow": series_now})
    # ts를 행으로 추가(값은 NaN) -> lag 계산 편의
    tmp_df.loc[ts, "activePow"] = np.nan
    tmp_df = tmp_df.sort_index()

    X_t = time_features(pd.DatetimeIndex([ts]))
    X_l = add_lag_rolling(tmp_df, "activePow").loc[[ts]]
    X_row = pd.concat([X_t, X_l], axis=1)[FEATURE_COLS]

    yhat = models["activePow"].predict(X_row)[0]
    yhat = max(float(yhat), 0.0)
    pred_active.append(yhat)

pred_sorted["activePow"] = pred_active

# 5-2) 나머지 컬럼 예측
#     - 동일한 FEATURE_COLS(시간 + activePow lag/rolling)로 예측
#     - 여기서 lag/rolling은 "예측된 activePow"를 포함한 전체(activePow series)로 계산
full_active_for_features = pd.concat([
    hist_active,
    pd.Series(pred_sorted["activePow"].values, index=pred_sorted.index, name="activePow")
]).sort_index()

tmp_all = pd.DataFrame({"activePow": full_active_for_features})

X_time_future = time_features(pred_sorted.index)
X_lag_future  = add_lag_rolling(tmp_all, "activePow").loc[pred_sorted.index]
X_feat_future = pd.concat([X_time_future, X_lag_future], axis=1)[FEATURE_COLS]

for t in TARGET_COLS:
    if t == "activePow":
        continue
    yhat = models[t].predict(X_feat_future)
    # 최소한의 물리 제약
    if t.startswith("voltage") or t.startswith("current"):
        yhat = np.maximum(yhat, 0)
    pred_sorted[t] = yhat

# =============================================================================
# 6) 출력 포맷 만들기: 열=rtu_15min 컬럼, 행=5month 행
# =============================================================================
canonical_cols = pd.read_csv(RTU_15MIN_PATH, nrows=0).columns.tolist()

# 출력 DF 생성(5month 원래 행 순서 기준)
df_out = pd.DataFrame(index=np.arange(len(df_rows)))
for c in canonical_cols:
    df_out[c] = np.nan

# datetime 채우기(원래 5month 형식을 최대한 유지: 원본 컬럼명이 datetime이 아니어도 출력은 datetime 컬럼을 우선 사용)
if "datetime" in df_out.columns:
    df_out["datetime"] = pd.to_datetime(dt_future).strftime("%Y-%m-%d %H:%M:%S")
else:
    # canonical에 datetime이 없으면 새로 만들어도 되지만, 보통 있음
    df_out["datetime"] = pd.to_datetime(dt_future).strftime("%Y-%m-%d %H:%M:%S")

# module(equipment)가 canonical에 있으면 TOTAL로
if MODULE_COL in df_out.columns:
    df_out[MODULE_COL] = "TOTAL"

# pred_sorted(시간정렬) -> 원래 순서로 되돌려 매핑
pred_sorted_reset = pred_sorted.reset_index().rename(columns={"__dt__": "datetime"})
pred_sorted_reset["datetime"] = pd.to_datetime(pred_sorted_reset["datetime"])
map_pred = pred_sorted_reset.set_index("datetime")

# 5month의 각 행 datetime에 해당하는 예측값을 채움
dt_future_series = pd.to_datetime(dt_future)
for t in TARGET_COLS:
    if t in df_out.columns:
        df_out[t] = map_pred.loc[dt_future_series, t].to_numpy()

# operation: 예측 안함 -> 1 고정(있으면)
if "operation" in df_out.columns:
    df_out["operation"] = 1

# 파생/누적(공장 전체 0부터)
df_out["kWh_from_activePow"] = df_out["activePow"] * STEP_HOURS

if "hourly_energy_used_kWh" in df_out.columns:
    # 컬럼명 유지, 값은 15분 kWh
    df_out["hourly_energy_used_kWh"] = df_out["kWh_from_activePow"]

if "accumActiveEnergy" in df_out.columns:
    # 원래 행 순서 보존용 인덱스
    df_out["_ord_"] = np.arange(len(df_out))

    # 시간 정렬 후 누적
    df_out["_dt_"] = pd.to_datetime(df_out["datetime"], errors="coerce")
    df_out = df_out.sort_values("_dt_").reset_index(drop=True)

    df_out["accumActiveEnergy"] = df_out["kWh_from_activePow"].cumsum()

    # 원래 행 순서로 복원
    df_out = df_out.sort_values("_ord_").reset_index(drop=True)
    df_out = df_out.drop(columns=["_ord_", "_dt_"])

# SumCurrent / MeanPowerFactor / 비용 / 탄소
if all(c in df_out.columns for c in ["currentR","currentS","currentT"]):
    df_out["SumCurrent"] = df_out["currentR"] + df_out["currentS"] + df_out["currentT"]

if all(c in df_out.columns for c in ["powerFactR","powerFactS","powerFactT"]):
    df_out["MeanPowerFactor"] = (df_out["powerFactR"] + df_out["powerFactS"] + df_out["powerFactT"]) / 3

if "electricityCost_won" in df_out.columns:
    df_out["electricityCost_won"] = df_out["kWh_from_activePow"] * PRICE_WON_PER_KWH

if "carbon_kgCO2" in df_out.columns:
    df_out["carbon_kgCO2"] = df_out["kWh_from_activePow"] * CARBON_KGCO2_PER_KWH

# 컬럼 순서 고정: canonical 우선 + 추가된 컬럼 뒤
final_cols = [c for c in canonical_cols if c in df_out.columns] + [c for c in df_out.columns if c not in canonical_cols]
df_out = df_out[final_cols]

safe_mkdir(OUTPUT_PATH)
df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("saved:", OUTPUT_PATH)
print("rows:", len(df_out), "| predicted targets:", TARGET_COLS)


saved: D:/2025-2/BDA/공모전/sample data/output/pred_total_rows_5month_cols_rtu15min.csv
rows: 2688 | predicted targets: ['activePow', 'voltageR', 'voltageS', 'voltageT', 'voltageRS', 'voltageST', 'voltageTR', 'currentR', 'currentS', 'currentT', 'powerFactR', 'powerFactS', 'powerFactT', 'reactiveP']


In [10]:
import pandas as pd

df = pd.read_csv("D:/2025-2/BDA/공모전/sample data/rtu_data_15min_v2.csv")
df["dt"] = pd.to_datetime(df["datetime"], errors="coerce")
one = df[df["dt"] == df["dt"].dropna().iloc[0]]
print(one[["module(equipment)", "activePow", "currentR", "voltageR"]].head(20))
print("activePow std across modules:", one["activePow"].std())


       module(equipment)    activePow   currentR    voltageR
0                1(PM-3)  2933.969222  17.745556  215.015944
14401         11(우측분전반1)  2927.200444  17.156444  215.048556
28802            12(4호기)  3020.900167  17.818278  215.263722
43203            13(3호기)  3033.983944  18.029667  214.951944
57604            14(2호기)  3059.478833  17.928389  214.673889
72005          15(예비건조기)  3065.433611  17.685500  214.814444
86406           16(호이스트)  2955.633778  16.383056  214.766333
100807           17(6호기)  3071.717389  17.647167  215.180556
115208        18(우측분전반2)  2985.206111  17.296778  215.012500
129609          2(L-1전등)  2984.140111  17.240556  214.573833
144010         3(분쇄기(2))  3026.471278  19.197500  215.100444
158411         4(분쇄기(1))  3020.322833  17.614889  215.009833
172812          5(좌측분전반)  2999.656889  18.221667  214.835722
activePow std across modules: 47.62930356027036


In [3]:
import os
import gc
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# (필수) 모델 라이브러리
# =========================
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =============================================================================
# 경로 설정 (본인 환경에 맞게 수정)
# =============================================================================
RTU_15MIN_PATH   = r"D:/2025-2/BDA/공모전/sample data/rtu_data_15min_v2.csv"
ROW_5MONTH_PATH  = r"D:/2025-2/BDA/공모전/sample data/rtu_data_5month_dynamic_fixed.csv"
OUTPUT_PATH      = r"D:/2025-2/BDA/공모전/sample data/output/pred_total_15min_best_family.csv"

PRICE_WON_PER_KWH = 180
CARBON_KGCO2_PER_KWH = 0.424
STEP_HOURS = 0.25  # 15분

DT_CANDIDATES = ("datetime", "id", "timestamp", "localtime")
MODULE_COL = "module(equipment)"  # rtu_15min에 있으면 출력은 TOTAL로 채움

# =============================================================================
# 유틸
# =============================================================================
def safe_mkdir(path: str):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)

def find_dt_col(df: pd.DataFrame) -> str:
    for c in DT_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError(f"시간 컬럼을 찾지 못했습니다. 후보={DT_CANDIDATES}, 실제={df.columns.tolist()}")

def to_dt_index(s: pd.Series) -> pd.DatetimeIndex:
    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        bad = int(dt.isna().sum())
        ex = s[dt.isna()].head(3).tolist()
        raise ValueError(f"datetime 파싱 실패 {bad}개. 예: {ex}")
    return pd.DatetimeIndex(dt)

def calculate_smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

def time_features(index_dt: pd.DatetimeIndex) -> pd.DataFrame:
    """15분 예측: minute + dow_sin/cos 포함"""
    df = pd.DataFrame(index=index_dt).copy()

    df["hour"] = df.index.hour
    df["minute"] = df.index.minute
    df["dayofweek"] = df.index.dayofweek
    df["day"] = df.index.day
    df["month"] = df.index.month
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

    df["minute_sin"] = np.sin(2*np.pi*df["minute"]/60)
    df["minute_cos"] = np.cos(2*np.pi*df["minute"]/60)

    df["day_sin"] = np.sin(2*np.pi*df["day"]/31)
    df["day_cos"] = np.cos(2*np.pi*df["day"]/31)

    df["dow_sin"] = np.sin(2*np.pi*df["dayofweek"]/7)
    df["dow_cos"] = np.cos(2*np.pi*df["dayofweek"]/7)

    return df

def add_lag_rolling_activepow(active_series: pd.Series) -> pd.DataFrame:
    """activePow 기반 상태 피처: lag/rolling"""
    out = pd.DataFrame(index=active_series.index)
    s = active_series

    out["activePow_lag1"] = s.shift(1)      # 15분 전
    out["activePow_lag4"] = s.shift(4)      # 1시간 전
    out["activePow_lag96"] = s.shift(96)    # 1일 전

    out["activePow_rmean4"] = s.rolling(4, min_periods=1).mean()
    out["activePow_rstd4"]  = s.rolling(4, min_periods=2).std()
    out["activePow_rmean96"] = s.rolling(96, min_periods=1).mean()
    return out

def infer_and_make_15min_index(dt_from_rows: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """
    5month 파일의 시간 간격이 15분이 아니면,
    min~max 구간을 15분으로 촘촘히 생성 (요청: 최종 output 15분)
    """
    dt_sorted = pd.DatetimeIndex(dt_from_rows).sort_values()
    if len(dt_sorted) < 3:
        # 최소 정보만 있으면 15분 생성 불가 -> 그대로 사용
        return pd.DatetimeIndex(dt_from_rows)

    diffs = pd.Series(dt_sorted).diff().dropna().dt.total_seconds()
    med = diffs.median()

    # 15분(900s) +/- 여유
    if abs(med - 900) <= 60:
        return pd.DatetimeIndex(dt_from_rows)  # 원래 행 그대로
    else:
        start = dt_sorted.min()
        end = dt_sorted.max()
        return pd.date_range(start=start, end=end, freq="15T")

def new_model_by_name(name: str):
    """activePow에서 선택된 모델 패밀리를 동일 파라미터로 재생성(타깃별 학습용)"""
    if name == "XGBoost":
        return xgb.XGBRegressor(
            n_estimators=1400, max_depth=8, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            n_jobs=-1, random_state=42
        )
    if name == "LightGBM":
        return lgb.LGBMRegressor(
            n_estimators=1800, max_depth=12, num_leaves=96, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            n_jobs=-1, random_state=42, verbose=-1
        )
    if name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=400, max_depth=18, n_jobs=-1, random_state=42
        )
    if name == "ExtraTrees":
        return ExtraTreesRegressor(
            n_estimators=700, max_depth=None, n_jobs=-1, random_state=42
        )
    raise ValueError(name)

# =============================================================================
# 1) rtu_15min(모듈별) -> 공장 전체 시계열 생성(집계 규칙 확정)
# =============================================================================
df_rtu = pd.read_csv(RTU_15MIN_PATH)
dt_col_rtu = find_dt_col(df_rtu)
df_rtu["__dt__"] = to_dt_index(df_rtu[dt_col_rtu])

# 숫자형 캐스팅
for c in df_rtu.columns:
    if c not in [dt_col_rtu, "__dt__", MODULE_COL]:
        df_rtu[c] = pd.to_numeric(df_rtu[c], errors="coerce")

# 집계 규칙(확정)
SUM_COLS  = [c for c in ["activePow", "reactiveP"] if c in df_rtu.columns]
MEAN_COLS = [c for c in [
    "currentR","currentS","currentT",
    "voltageR","voltageS","voltageT",
    "voltageRS","voltageST","voltageTR",
    "powerFactR","powerFactS","powerFactT"
] if c in df_rtu.columns]

agg = {c: "sum" for c in SUM_COLS}
agg.update({c: "mean" for c in MEAN_COLS})

df_total = (
    df_rtu.groupby("__dt__", as_index=False).agg(agg)
          .sort_values("__dt__")
          .set_index("__dt__").sort_index()
          .resample("15T").mean()
)

if "activePow" not in df_total.columns:
    raise KeyError("activePow가 공장 전체 시계열에 없습니다. 원본 컬럼명 확인이 필요합니다.")

# =============================================================================
# 2) 피처 구성 (15분 + Full + activePow lag/rolling)
# =============================================================================
X_time = time_features(df_total.index)
X_lag  = add_lag_rolling_activepow(df_total["activePow"])
X_all  = pd.concat([X_time, X_lag], axis=1)

FEATURE_COLS = X_all.columns.tolist()

TARGETS = [c for c in [
    "activePow",
    "voltageR","voltageS","voltageT",
    "voltageRS","voltageST","voltageTR",
    "currentR","currentS","currentT",
    "powerFactR","powerFactS","powerFactT",
    "reactiveP"
] if c in df_total.columns]

# =============================================================================
# 3) activePow로만 "모델 패밀리" 선택 (원하는 2번 요구)
#    - val: 4월 (없으면 마지막 20%)
# =============================================================================
data_ap = pd.concat([X_all, df_total["activePow"].rename("y")], axis=1).dropna()
X_ap = data_ap[FEATURE_COLS]
y_ap = data_ap["y"]

if ("month" in X_ap.columns) and (X_ap["month"] == 4).any() and (X_ap["month"] != 4).any():
    tr = X_ap["month"] != 4
    va = X_ap["month"] == 4
    X_tr, y_tr = X_ap[tr], y_ap[tr]
    X_va, y_va = X_ap[va], y_ap[va]
else:
    n = len(X_ap)
    cut = int(n * 0.8)
    X_tr, y_tr = X_ap.iloc[:cut], y_ap.iloc[:cut]
    X_va, y_va = X_ap.iloc[cut:], y_ap.iloc[cut:]

CAND_NAMES = ["XGBoost", "LightGBM", "RandomForest", "ExtraTrees"]
best_name = None
best_smape = np.inf

for name in CAND_NAMES:
    m = new_model_by_name(name)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_va)
    smape = calculate_smape(y_va, pred)
    if smape < best_smape:
        best_smape = smape
        best_name = name

print(f"[activePow] Best model family = {best_name} | SMAPE={best_smape:.4f}")

# =============================================================================
# 4) 선택된 모델 패밀리(best_name)로 모든 타깃 학습
# =============================================================================
models = {}
for t in TARGETS:
    data_t = pd.concat([X_all, df_total[t].rename("y")], axis=1).dropna()
    X_t = data_t[FEATURE_COLS]
    y_t = data_t["y"]

    m = new_model_by_name(best_name)
    m.fit(X_t, y_t)
    models[t] = m
    gc.collect()

# =============================================================================
# 5) 5월 output 시간 인덱스 생성(최종은 15분)
# =============================================================================
df_rows = pd.read_csv(ROW_5MONTH_PATH)
dt_col_rows = find_dt_col(df_rows)
dt_rows = to_dt_index(df_rows[dt_col_rows])

future_index = infer_and_make_15min_index(dt_rows)
future_index = pd.DatetimeIndex(future_index).sort_values()

# =============================================================================
# 6) 예측 (activePow는 재귀로 lag 업데이트)
# =============================================================================
hist_active = df_total["activePow"].copy()

# 6-1) activePow 재귀 예측
pred_active = []
model_ap = models["activePow"]

for ts in future_index:
    series_now = pd.concat([hist_active, pd.Series(pred_active, index=future_index[:len(pred_active)])])
    series_now = series_now.sort_index()

    # 현재 시점 행을 포함해 lag 계산
    tmp = series_now.copy()
    tmp.loc[ts] = np.nan
    tmp = tmp.sort_index()

    X_t = time_features(pd.DatetimeIndex([ts]))
    X_l = add_lag_rolling_activepow(tmp).loc[[ts]]
    X_row = pd.concat([X_t, X_l], axis=1)[FEATURE_COLS]

    yhat = float(model_ap.predict(X_row)[0])
    yhat = max(yhat, 0.0)  # kW 음수 방지
    pred_active.append(yhat)

pred_df = pd.DataFrame(index=future_index)
pred_df["activePow"] = pred_active

# 6-2) 다른 컬럼 예측용 피처 구성(예측 activePow 포함)
full_active = pd.concat([
    hist_active,
    pd.Series(pred_df["activePow"].values, index=future_index, name="activePow")
]).sort_index()

X_time_f = time_features(future_index)
X_lag_f  = add_lag_rolling_activepow(full_active).loc[future_index]
X_feat_f = pd.concat([X_time_f, X_lag_f], axis=1)[FEATURE_COLS]

for t in TARGETS:
    if t == "activePow":
        continue
    yhat = models[t].predict(X_feat_f)

    # 최소 물리 제약
    if t.startswith("voltage") or t.startswith("current"):
        yhat = np.maximum(yhat, 0)

    pred_df[t] = yhat

# =============================================================================
# 7) 출력 포맷: 열=rtu_15min 컬럼, 행=15분(5월)
# =============================================================================
canonical_cols = pd.read_csv(RTU_15MIN_PATH, nrows=0).columns.tolist()

df_out = pd.DataFrame(index=np.arange(len(future_index)))
for c in canonical_cols:
    df_out[c] = np.nan

# datetime 채우기
if "datetime" in df_out.columns:
    df_out["datetime"] = future_index.strftime("%Y-%m-%d %H:%M:%S")
else:
    df_out["datetime"] = future_index.strftime("%Y-%m-%d %H:%M:%S")

# module 컬럼이 있으면 TOTAL
if MODULE_COL in df_out.columns:
    df_out[MODULE_COL] = "TOTAL"

# 예측값 주입
for t in TARGETS:
    if t in df_out.columns:
        df_out[t] = pred_df[t].to_numpy()

# operation: 예측 안함
if "operation" in df_out.columns:
    df_out["operation"] = 1

# 파생/누적
df_out["kWh_from_activePow"] = df_out["activePow"] * STEP_HOURS

if "hourly_energy_used_kWh" in df_out.columns:
    # 컬럼명 유지, 값은 15분 kWh
    df_out["hourly_energy_used_kWh"] = df_out["kWh_from_activePow"]

if "accumActiveEnergy" in df_out.columns:
    # 0부터 시작 누적
    df_out["_ord_"] = np.arange(len(df_out))
    df_out["_dt_"] = pd.to_datetime(df_out["datetime"])
    df_out = df_out.sort_values("_dt_").reset_index(drop=True)
    df_out["accumActiveEnergy"] = df_out["kWh_from_activePow"].cumsum()
    df_out = df_out.sort_values("_ord_").reset_index(drop=True)
    df_out = df_out.drop(columns=["_ord_", "_dt_"])

# 부가 파생(있을 때만 채움)
if all(c in df_out.columns for c in ["currentR","currentS","currentT"]):
    df_out["SumCurrent"] = df_out["currentR"] + df_out["currentS"] + df_out["currentT"]

if all(c in df_out.columns for c in ["powerFactR","powerFactS","powerFactT"]):
    df_out["MeanPowerFactor"] = (df_out["powerFactR"] + df_out["powerFactS"] + df_out["powerFactT"]) / 3

if "electricityCost_won" in df_out.columns:
    df_out["electricityCost_won"] = df_out["kWh_from_activePow"] * PRICE_WON_PER_KWH

if "carbon_kgCO2" in df_out.columns:
    df_out["carbon_kgCO2"] = df_out["kWh_from_activePow"] * CARBON_KGCO2_PER_KWH

# 컬럼 순서 고정: canonical 우선 + 추가 컬럼 뒤
final_cols = [c for c in canonical_cols if c in df_out.columns] + [c for c in df_out.columns if c not in canonical_cols]
df_out = df_out[final_cols]

safe_mkdir(OUTPUT_PATH)
df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("saved:", OUTPUT_PATH)
print("rows:", len(df_out), "| freq=15min | model_family:", best_name, "| targets:", TARGETS)


[activePow] Best model family = RandomForest | SMAPE=0.3363
saved: D:/2025-2/BDA/공모전/sample data/output/pred_total_15min_best_family.csv
rows: 2688 | freq=15min | model_family: RandomForest | targets: ['activePow', 'voltageR', 'voltageS', 'voltageT', 'voltageRS', 'voltageST', 'voltageTR', 'currentR', 'currentS', 'currentT', 'powerFactR', 'powerFactS', 'powerFactT', 'reactiveP']
